In [ ]:
!pip install seaborn
!pip install -U scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

class ProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        x = self.mlp(x)
        return F.normalize(x, p=2, dim=1)

class MultiModalAlignmentModel(nn.Module):
    def __init__(self, img_in_dim=512, graph_in_dim=512, shared_dim=256):
        super().__init__()
        self.image_proj = ProjectionHead(input_dim=img_in_dim, hidden_dim=512, output_dim=shared_dim)
        self.graph_proj = ProjectionHead(input_dim=graph_in_dim, hidden_dim=512, output_dim=shared_dim)

    def forward(self, img_embeds, graph_embeds):
        return self.image_proj(img_embeds), self.graph_proj(graph_embeds)

print("Loading data and model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load raw embeddings
raw_images = np.load('image_embeddings.npy')
raw_graphs = np.load('graph_embeddings.npy')

# Load Trained Model
model = MultiModalAlignmentModel(img_in_dim=512, graph_in_dim=512, shared_dim=256).to(device)
model.load_state_dict(torch.load('multimodal_projection_heads.pth', map_location=device))
model.eval()

num_samples = 1500
indices = np.random.choice(raw_images.shape[0], num_samples, replace=False)

sample_raw_images = torch.tensor(raw_images[indices], dtype=torch.float32).to(device)
sample_raw_graphs = torch.tensor(raw_graphs[indices], dtype=torch.float32).to(device)

with torch.no_grad():
    proj_images, proj_graphs = model(sample_raw_images, sample_raw_graphs)
    
    proj_images = proj_images.cpu().numpy()
    proj_graphs = proj_graphs.cpu().numpy()
    
sample_raw_images = sample_raw_images.cpu().numpy()
sample_raw_graphs = sample_raw_graphs.cpu().numpy()

combined_raw = np.vstack((sample_raw_images, sample_raw_graphs))
combined_proj = np.vstack((proj_images, proj_graphs))

labels = ['Image Modality'] * num_samples + ['Graph Modality'] * num_samples

# Run t-SNE
tsne_raw = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(combined_raw)
tsne_proj = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(combined_proj)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Before Contrastive Learning
sns.scatterplot(
    x=tsne_raw[:, 0], y=tsne_raw[:, 1],
    hue=labels, palette=['#1f77b4', '#ff7f0e'],
    alpha=0.6, s=30, ax=axes[0]
)
axes[0].set_title("BEFORE Contrastive Learning\n(Raw CLIP vs Raw Node2Vec)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("t-SNE Dimension 1")
axes[0].set_ylabel("t-SNE Dimension 2")

# Plot 2: After Contrastive Learning
sns.scatterplot(
    x=tsne_proj[:, 0], y=tsne_proj[:, 1],
    hue=labels, palette=['#1f77b4', '#ff7f0e'],
    alpha=0.6, s=30, ax=axes[1]
)
axes[1].set_title("AFTER Contrastive Learning\n(Unified Shared Space)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("t-SNE Dimension 1")
axes[1].set_ylabel("t-SNE Dimension 2")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ProjectionHeadPro(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout_rate),   
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.mlp(x)
        return F.normalize(x, p=2, dim=1)

class MultiModalAlignmentModelPro(nn.Module):
    def __init__(self, img_in_dim=512, graph_in_dim=512, shared_dim=256):
        super().__init__()
        self.image_proj = ProjectionHeadPro(input_dim=img_in_dim, hidden_dim=512, output_dim=shared_dim)
        self.graph_proj = ProjectionHeadPro(input_dim=graph_in_dim, hidden_dim=512, output_dim=shared_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, img_embeds, graph_embeds):
        return self.image_proj(img_embeds), self.graph_proj(graph_embeds)

# LOAD THE MODEL AND WEIGHTS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_pro = MultiModalAlignmentModelPro(img_in_dim=512, graph_in_dim=512, shared_dim=256).to(device)

model_pro.load_state_dict(torch.load('multimodal_projection_heads_v2.pth', map_location=device))

model_pro.eval()

# LOAD RAW DATA & PROJECT INTO SHARED SPACE
raw_images = torch.tensor(np.load('image_embeddings.npy'), dtype=torch.float32).to(device)
raw_graphs = torch.tensor(np.load('graph_embeddings.npy'), dtype=torch.float32).to(device)

with torch.no_grad():
    shared_images = model_pro.image_proj(raw_images)
    shared_graphs = model_pro.graph_proj(raw_graphs)
    
    shared_images = shared_images.cpu().numpy()
    shared_graphs = shared_graphs.cpu().numpy()

# CROSS-MODAL EVALUATION (Image -> Graph)
cross_modal_scores = shared_images.dot(shared_graphs.T)

def calculate_recall_at_k(similarity_matrix, k_values=[1, 5, 10, 50]):
    N = similarity_matrix.shape[0]
    
    top_k_indices = np.argsort(similarity_matrix, axis=1)[:, ::-1]
    
    recalls = {}
    for k in k_values:
        correct_retrievals = 0
        for i in range(N):
            if i in top_k_indices[i, :k]:
                correct_retrievals += 1
                
        recalls[f'Recall@{k}'] = (correct_retrievals / N) * 100
    
    return recalls

recalls = calculate_recall_at_k(cross_modal_scores, k_values=[1, 5, 10, 50, 100])

for metric, score in recalls.items():
    print(f"{metric}: {score:.2f}%")